# 05 — SQL Business Analysis (Gold Layer)

20 business-focused SQL queries over the governed Gold datasets in `dev.gold` (`fact_sales`, `dim_customer`, `dim_product`, `dim_supplier`, `dim_geography`, `vw_annual_sales_kpis`), covering revenue KPIs, trends, customer/product/supplier/geography performance, shipping and order behaviour.

**Metric definitions (used consistently):**
- **Net revenue (before tax)** = `SUM(net_revenue)` where `net_revenue = extended_price × (1 − discount_rate)`
- **Orders** = `COUNT(DISTINCT order_key)`
- **Active customers** = `COUNT(DISTINCT customer_key)` with at least one order in the period
- **Average order value** = net revenue ÷ distinct orders

Each query is followed by a business interpretation and a suggested management action.

In [0]:
%sql
-- Q01: Overall KPI summary — net revenue, orders, active customers, average order value
SELECT
  ROUND(SUM(net_revenue), 2)                AS net_revenue,
  COUNT(DISTINCT order_key)                  AS total_orders,
  COUNT(DISTINCT customer_key)               AS active_customers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM dev.gold.fact_sales;

In [0]:
%sql
-- Q02: Monthly revenue trend across all years
SELECT
  date_format(order_date, 'yyyy-MM') AS order_month,
  ROUND(SUM(net_revenue), 2)         AS net_revenue
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
-- Q03: Annual revenue with year-over-year change
WITH annual AS (
  SELECT order_year, ROUND(SUM(net_revenue), 2) AS net_revenue
  FROM dev.gold.fact_sales
  GROUP BY 1
)
SELECT
  order_year,
  net_revenue,
  LAG(net_revenue) OVER (ORDER BY order_year)                        AS prev_year_revenue,
  ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY order_year), 2) AS yoy_abs_change,
  ROUND(100 * (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year))
        / LAG(net_revenue) OVER (ORDER BY order_year), 2)            AS yoy_pct_change
FROM annual
ORDER BY order_year;

In [0]:
%sql
-- Q04: Revenue by customer market segment
SELECT
  customer_market_segment,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Q05: Top 10 customers by net revenue
SELECT
  customer_key,
  customer_name,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT order_key)  AS orders
FROM dev.gold.fact_sales
GROUP BY 1, 2
ORDER BY net_revenue DESC
LIMIT 10;

In [0]:
%sql
-- Q06: Top 10 customers as a share of total revenue
WITH top10 AS (
  SELECT SUM(net_revenue) AS top10_revenue
  FROM (
    SELECT SUM(net_revenue) AS net_revenue
    FROM dev.gold.fact_sales
    GROUP BY customer_key
    ORDER BY net_revenue DESC
    LIMIT 10
  )
),
total AS (
  SELECT SUM(net_revenue) AS total_revenue FROM dev.gold.fact_sales
)
SELECT
  ROUND(t.top10_revenue, 2)                    AS top10_customer_revenue,
  ROUND(tot.total_revenue, 2)                  AS total_revenue,
  ROUND(100 * t.top10_revenue / tot.total_revenue, 2) AS top10_share_pct
FROM top10 t CROSS JOIN total tot;

In [0]:
%sql
-- Q07: Top 10 products by net revenue
SELECT
  product_key,
  product_name,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  SUM(quantity)              AS units_sold
FROM dev.gold.fact_sales
GROUP BY 1, 2
ORDER BY net_revenue DESC
LIMIT 10;

In [0]:
%sql
-- Q08: Bottom 10 products by net revenue
SELECT
  product_key,
  product_name,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  SUM(quantity)              AS units_sold
FROM dev.gold.fact_sales
GROUP BY 1, 2
ORDER BY net_revenue ASC
LIMIT 10;

In [0]:
%sql
-- Q09: Revenue by product type
SELECT
  product_type,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Q10: Top 10 suppliers by net revenue
SELECT
  supplier_key,
  supplier_name,
  supplier_nation_name,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM dev.gold.fact_sales
GROUP BY 1, 2, 3
ORDER BY net_revenue DESC
LIMIT 10;

In [0]:
%sql
-- Q11: Revenue share of the top 10 suppliers
WITH top10 AS (
  SELECT SUM(net_revenue) AS top10_revenue
  FROM (
    SELECT SUM(net_revenue) AS net_revenue
    FROM dev.gold.fact_sales
    GROUP BY supplier_key
    ORDER BY net_revenue DESC
    LIMIT 10
  )
),
total AS (
  SELECT SUM(net_revenue) AS total_revenue FROM dev.gold.fact_sales
)
SELECT
  ROUND(t.top10_revenue, 2)                          AS top10_supplier_revenue,
  ROUND(tot.total_revenue, 2)                         AS total_revenue,
  ROUND(100 * t.top10_revenue / tot.total_revenue, 2) AS top10_share_pct
FROM top10 t CROSS JOIN total tot;

In [0]:
%sql
-- Q12: Revenue by customer region
SELECT
  customer_region_name,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  COUNT(DISTINCT customer_key) AS active_customers,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Q13: Revenue by customer nation (all nations)
SELECT
  customer_nation_name,
  customer_region_name,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1, 2
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Q14: Revenue and volume split by return flag
SELECT
  return_flag,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  SUM(quantity)              AS units_sold,
  COUNT(*)                  AS order_lines,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Q15: Average discount by market segment
SELECT
  customer_market_segment,
  ROUND(AVG(discount_rate), 4) AS avg_discount_rate,
  ROUND(AVG(discount_rate) * 100, 2) AS avg_discount_pct
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY avg_discount_rate DESC;

In [0]:
%sql
-- Q16: Average ship-to-receipt days by shipping mode
SELECT
  ship_mode,
  ROUND(AVG(DATEDIFF(receipt_date, ship_date)), 2) AS avg_ship_to_receipt_days,
  COUNT(*) AS order_lines
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY avg_ship_to_receipt_days DESC;

In [0]:
%sql
-- Q17: Order count and revenue by order priority
SELECT
  order_priority,
  COUNT(DISTINCT order_key)  AS orders,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Q18: Monthly breakdown of order status
SELECT
  date_format(order_date, 'yyyy-MM') AS order_month,
  order_status,
  COUNT(DISTINCT order_key) AS orders
FROM dev.gold.fact_sales
GROUP BY 1, 2
ORDER BY 1, 2;

In [0]:
%sql
-- Q19: Product revenue ranking within each market segment (top 3 per segment)
WITH product_segment AS (
  SELECT
    customer_market_segment,
    product_key,
    product_name,
    SUM(net_revenue) AS net_revenue
  FROM dev.gold.fact_sales
  GROUP BY 1, 2, 3
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY customer_market_segment ORDER BY net_revenue DESC) AS revenue_rank
  FROM product_segment
)
SELECT
  customer_market_segment,
  revenue_rank,
  product_key,
  product_name,
  ROUND(net_revenue, 2) AS net_revenue
FROM ranked
WHERE revenue_rank <= 3
ORDER BY customer_market_segment, revenue_rank;

In [0]:
%sql
-- Q20: Revenue comparison of customer geography vs supplier geography by region
WITH customer_side AS (
  SELECT
    customer_region_key AS region_key,
    customer_region_name AS region_name,
    SUM(net_revenue) AS customer_region_revenue
  FROM dev.gold.fact_sales
  GROUP BY 1, 2
),
supplier_side AS (
  SELECT
    supplier_region_key AS region_key,
    supplier_region_name AS region_name,
    SUM(net_revenue) AS supplier_region_revenue
  FROM dev.gold.fact_sales
  GROUP BY 1, 2
)
SELECT
  COALESCE(c.region_name, s.region_name)                    AS region_name,
  ROUND(c.customer_region_revenue, 2)                        AS customer_side_revenue,
  ROUND(s.supplier_region_revenue, 2)                        AS supplier_side_revenue,
  ROUND(s.supplier_region_revenue - c.customer_region_revenue, 2) AS supply_minus_demand
FROM customer_side c
FULL OUTER JOIN supplier_side s
  ON c.region_key = s.region_key
ORDER BY region_name;